In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/all_bank_reviews.csv")
df.head()

,review_id,user_name,review,rating,thumbs_up,review_date,bank,clean_review
0,eb3cc438-1c10-4e72-8851-3efff6a04135,Hamza Jemal,this app very full,5,0,2026-05-16 09:17:00,CBE,thisappveryfull
1,f8209985-ea16-4f28-bb48-d6a7276f0f08,Ras Abale,good apps,4,0,2026-05-16 07:18:33,CBE,goodapps
2,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,Abrha Hadgu,this update got crazy i don't know what's goin...,1,0,2026-05-15 23:20:32,CBE,thisupdategotcrazyidontknowwhatsgoingonthisapp...
3,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,Ademasu Shadaga,thanks for you 😘,5,0,2026-05-15 20:11:22,CBE,thanksforyou
4,7019e213-93dc-4f00-bff3-80cfb80e5d3a,suutu sura89,it's okay,4,0,2026-05-15 19:53:26,CBE,itsokay


In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(str(text))['compound']
    
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

In [3]:
df["sentiment"] = df["review"].apply(get_sentiment)
df.head()

,review_id,user_name,review,rating,thumbs_up,review_date,bank,clean_review,sentiment
0,eb3cc438-1c10-4e72-8851-3efff6a04135,Hamza Jemal,this app very full,5,0,2026-05-16 09:17:00,CBE,thisappveryfull,neutral
1,f8209985-ea16-4f28-bb48-d6a7276f0f08,Ras Abale,good apps,4,0,2026-05-16 07:18:33,CBE,goodapps,positive
2,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,Abrha Hadgu,this update got crazy i don't know what's goin...,1,0,2026-05-15 23:20:32,CBE,thisupdategotcrazyidontknowwhatsgoingonthisapp...,positive
3,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,Ademasu Shadaga,thanks for you 😘,5,0,2026-05-15 20:11:22,CBE,thanksforyou,positive
4,7019e213-93dc-4f00-bff3-80cfb80e5d3a,suutu sura89,it's okay,4,0,2026-05-15 19:53:26,CBE,itsokay,positive


In [5]:
df["sentiment"].value_counts()

sentiment
positive    1017
neutral      462
negative     339
Name: count, dtype: int64

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50, stop_words="english")

X = vectorizer.fit_transform(df["review"].astype(str))

In [7]:
keywords = vectorizer.get_feature_names_out()
keywords[:20]

array(['account', 'amazing', 'app', 'application', 'apps', 'bad', 'bank',
       'banking', 'best', 'better', 'boa', 'cbe', 'dashen', 'doesn',
       'don', 'easy', 'ethiopia', 'excellent', 'experience', 'fast'],
      dtype=object)

In [8]:
def assign_theme(text):
    text = str(text).lower()

    if "login" in text or "password" in text:
        return "Login Issues"
    elif "transfer" in text or "send" in text:
        return "Transaction Performance"
    elif "otp" in text or "code" in text:
        return "OTP Issues"
    elif "crash" in text or "error" in text:
        return "App Stability"
    elif "ui" in text or "design" in text:
        return "User Interface"
    else:
        return "Other"

In [9]:
df["theme"] = df["review"].apply(assign_theme)
df.head()

,review_id,user_name,review,rating,thumbs_up,review_date,bank,clean_review,sentiment,theme
0,eb3cc438-1c10-4e72-8851-3efff6a04135,Hamza Jemal,this app very full,5,0,2026-05-16 09:17:00,CBE,thisappveryfull,neutral,Other
1,f8209985-ea16-4f28-bb48-d6a7276f0f08,Ras Abale,good apps,4,0,2026-05-16 07:18:33,CBE,goodapps,positive,Other
2,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,Abrha Hadgu,this update got crazy i don't know what's goin...,1,0,2026-05-15 23:20:32,CBE,thisupdategotcrazyidontknowwhatsgoingonthisapp...,positive,Other
3,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,Ademasu Shadaga,thanks for you 😘,5,0,2026-05-15 20:11:22,CBE,thanksforyou,positive,Other
4,7019e213-93dc-4f00-bff3-80cfb80e5d3a,suutu sura89,it's okay,4,0,2026-05-15 19:53:26,CBE,itsokay,positive,Other


In [10]:
df.to_csv("../data/processed/task2_sentiment_themes.csv", index=False)

In [11]:
df.groupby("bank")["sentiment"].value_counts()

bank    sentiment
BOA     positive     261
        neutral      192
        negative     155
CBE     positive     328
        neutral      144
        negative      97
Dashen  positive     428
        neutral      126
        negative      87
Name: count, dtype: int64